In [1]:
from openai import OpenAI
import requests
import json

client = OpenAI()

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

In [3]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return response.json()


def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    response.raise_for_status()
    return response.json()

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    response.raise_for_status()
    return response.json()

In [4]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the current popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a movie by movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get similar movies by movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [7]:
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [8]:
messages = [
    {
        "role": "system",
        "content": """
You are a helpful movie expert agent.

Use tools when you need real movie data.
Remember previous conversation context.
If the user says "that movie", "it", "similar movies", or asks a follow-up question,
use the previous conversation to infer which movie they mean.

Answer in Korean.
"""
    }
]

In [9]:
def call_ai():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
        )

        message = response.choices[0].message

        if not message.tool_calls:
            messages.append({
                "role": "assistant",
                "content": message.content,
            })

            print(f"Agent: {message.content}")
            break

        messages.append({
            "role": "assistant",
            "content": message.content,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
                for tool_call in message.tool_calls
            ],
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments or "{}")

            print(f"[Tool Call] {function_name}({arguments})")

            function_to_run = FUNCTION_MAP[function_name]
            result = function_to_run(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result, ensure_ascii=False),
            })

In [11]:
def chat(user_message):
    messages.append({
        "role": "user",
        "content": user_message,
    })

    print(f"User: {user_message}")
    call_ai()
    print()

In [ ]:
chat("지금 인기 있는 영화 알려줘")

User: 지금 인기 있는 영화 알려줘
[Tool Call] get_popular_movies({})
Agent: 현재 인기 있는 영화는 다음과 같습니다:

1. **Obsession**
   - 개봉일: 2026-05-13
   - 평점: 7.9
   - 줄거리: "One Wish Willow"를 부수고 자신의 짝사랑을 쟁취하려는 로맨틱한 남자가 욕망의 어두운 가격을 발견하게 되는 이야기입니다.
   - ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**
   - 개봉일: 2026-06-03
   - 평점: 6.5
   - 줄거리: 1980년대 안드라 프라데시의 시골에서, 한 열정적인 마을 사람이 스포츠를 통해 지역 사회를 단결시키고 강력한 경쟁자에 맞서 자존심을 지키는 이야기입니다.
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**
   - 개봉일: 2026-06-04
   - 평점: 5.5
   - 줄거리: 결혼을 포기하고 새로운 해외 로맨스를 시작한 주인공이 충격적인 진실로 사랑과 충성, 그리고 진정한 약속의 의미를 직면하게 되는 이야기입니다.
   - ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man**
   - 개봉일: 2021-10-16
   - 평점: 8.2
   - 줄거리: 플란드르 작가 루이스가 영감을 찾기 위해 코트 다주르에 고립하려는 이야기입니다.
   - ![The Unknown Man](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaYOLF8Q3haz.jpg)

5. *

In [13]:
chat("듄에 대해 더 알려줘")

User: 듄에 대해 더 알려줘
[Tool Call] get_movie_details({'id': 1567})
Agent: "듄"에 대한 정보는 다음과 같습니다:

- **제목**: I Stand Alone (원제: Seul contre tous)
- **개봉일**: 1999-02-17
- **장르**: 드라마, 스릴러
- **러닝 타임**: 89분
- **평점**: 7.4 (투표 수: 576)
- **줄거리**: 정신적으로 장애가 있는 10대 딸을 유혹하려는 남성을 폭행한 후 감옥에서 출소한 정육점 주인은 새롭게 삶을 시작하고자 합니다. 그는 딸을 기관에 맡기고 파리 근교로 이사와 자신의 여자친구와 함께 새로운 정육점 가게를 열겠다고 약속을 받습니다. 그러나 그녀가 거짓말을 했다는 것을 알게 된 정육점 주인은 자신의 딸을 찾기 위해 다시 파리로 돌아갑니다.
- **제작국**: 프랑스
- **제작사**: Les Cinémas de la Zone, Love Streams Productions
- **예고편 및 홈페이지**: 정보 없음
- **태그라인**: "In the bowels of France."
- **포스터**: 
  ![I Stand Alone](https://image.tmdb.org/t/p/w780/4kMB2AAcDQ5diRuh7NTxuS90bx7.jpg)

더 궁금한 점이 있으면 말씀해 주세요!



In [14]:
chat("비슷한 영화 추천해 줄래?")

User: 비슷한 영화 추천해 줄래?
[Tool Call] get_similar_movies({'id': 1567})
Agent: "듄"과 비슷한 영화를 추천해 드리겠습니다:

1. **Captive**
   - 개봉일: 1986-09-19
   - 평점: 4.0
   - 줄거리: 국제 재산가의 딸이 납치되어 세 남자에게 길러지면서 벌어지는 이야기입니다.
   - ![Captive](https://image.tmdb.org/t/p/w780/ygq0vO3I0I7HTzmZVRNLIUdIzbV.jpg)

2. **Coyote**
   - 개봉일: 1992-06-26
   - 평점: 4.2
   - 줄거리: 세인트 로렌스 강 입구에서 일어나는 첫 사랑의 이야기입니다.
   - ![Coyote](https://image.tmdb.org/t/p/w780/n62dvrS6Y9XPUucxW7ulgVyyXWv.jpg)

3. **Sno-Line**
   - 개봉일: 1985-09-13
   - 평점: 7.0
   - 줄거리: 뉴욕의 지방 검사에게 배경이 되는 이야기를 다룬 영화입니다.
   - ![Sno-Line](https://image.tmdb.org/t/p/w780/w7l9QfULz2seQt6Tk3h3zUdTxHd.jpg)

4. **The Dark**
   - 개봉일: 2005-09-28
   - 평점: 5.7
   - 줄거리: 가족을 모으기 위해 여행을 떠난 여자가 딸을 잃고 미스터리한 사건이 벌어지는 이야기입니다.
   - ![The Dark](https://image.tmdb.org/t/p/w780/h6gCVAzjFLhzkffv2VZO1RgVCBt.jpg)

5. **The Resemblance**
   - 개봉일: 2022-06-09
   - 평점: 5.9
   - 줄거리: 유가족이 아들의 역할을 맡은 배우를 고용하려고 할 때 발생하는 사건에 대한 이야기입니다.
   - ![The Resemblance](https://image.tmdb.org/t/p/w780/2